In [0]:
"""Scans the uc_functions/ directory and outputs a JSON array of function files
for the forEach task to iterate over.

Derives the uc_functions/ path from this notebook's own location (sibling dir),
with an optional workspace_path override for non-source-linked deploys.

Outputs:
  Task value 'function_files' — JSON array of objects with 'function_file' key
  Task value 'uc_functions_path' — resolved workspace path for downstream tasks
"""
import os
import json

# Derive path from this notebook's location (works in source-linked and standard deploys)
try:
    workspace_path = dbutils.widgets.get("workspace_path")
except Exception:
    workspace_path = ""

if not workspace_path:
    # Fallback: derive from this notebook's own workspace path
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    # nb_path = .../src/deploy/discover_uc_functions
    # uc_functions/ is a sibling directory
    parent_dir = os.path.dirname(nb_path)
    workspace_path = os.path.join(parent_dir, "uc_functions")
    print(f"Derived workspace_path from notebook location: {workspace_path}")

# On serverless, workspace files are accessible via /Workspace/ prefix
fs_path = f"/Workspace{workspace_path}" if not workspace_path.startswith("/Workspace") else workspace_path

print(f"Scanning: {fs_path}")

sql_files = sorted([
    f for f in os.listdir(fs_path)
    if f.endswith(".sql") and not f.startswith("_")
])

print(f"Discovered {len(sql_files)} UC function templates:")
for f in sql_files:
    print(f"  - {f}")

# Output as task values for forEach iteration
forEach_inputs = [{"function_file": f} for f in sql_files]

dbutils.jobs.taskValues.set(key="function_files", value=json.dumps(forEach_inputs))
dbutils.jobs.taskValues.set(key="uc_functions_path", value=workspace_path)
print(f"\nTask values set: {len(forEach_inputs)} function files, path='{workspace_path}'")